# Stage 3: Inference, Validation Scoring and Submission Generation
This notebook runs the final evaluation predictions over validation set and generates leaderboard-ready `submission.csv` over test set using Minimum Bayes Risk reranking.

### 1. Install & Load Dependencies

In [ ]:
%pip install -q bert_score rouge_score pandas transformers accelerate datasets

import os
import sys
import pandas as pd
import torch

sys.path.append(os.path.abspath('src'))
import inference_utils
import metric_utils

print("Libraries loaded. CUDA Available:", torch.cuda.is_available())

### 2. Load Final Merged Model for Inference

In [ ]:
model_path = "/kaggle/working/final_model"
model, tokenizer = inference_utils.load_model_for_inference(model_path)

### 3. RUN VALIDATION MODE
Generates responses over the held-out validation dataset `sft_val.csv` using MBR decoding.

In [ ]:
val_path = "/kaggle/working/sft_val.csv"
val_df = pd.read_csv(val_path)

print("Starting validation generation (first 100 rows for fast checking, modify code to score all if time permits)... ")
# Limit to 100 rows for validation checks in SFT validation loop to keep runtime low
val_subset = val_df.head(100).copy()

val_preds_df = inference_utils.run_inference(model, tokenizer, val_subset, k=4)

# Save validation predictions
val_preds_df.to_csv("/kaggle/working/val_predictions.csv", index=False)
print("Saved validation predictions to /kaggle/working/val_predictions.csv")

### 4. Score Validation Predictions locally

In [ ]:
# Score validation predictions using the metric harness
try:
    val_score = metric_utils.score_predictions_csv(
        pred_csv_path="/kaggle/working/val_predictions.csv",
        ref_csv_path=val_path,
        id_col="id",
        pred_col="output",
        ref_col="output"
    )
    print(f"Local Validation Composite Score: {val_score:.4f}")
except Exception as e:
    print(f"Error scoring validation set: {e}")
    val_score = 0.0

### 5. Validation Quality Sanity Check Gate
A gate to prevent submitting broken predictions if the composite score indicates a parsing/decoding bug.

In [ ]:
if val_score < 0.3:
    print("\n[CRITICAL WARN] Local Validation Composite Score is abnormally low (< 0.3).")
    print("This suggests format leakage, truncation, or encoding issues.")
    # Print first 3 raw examples for review
    preds_df = pd.read_csv("/kaggle/working/val_predictions.csv")
    for idx, row in preds_df.head(3).iterrows():
        print(f"\nSample {idx+1}:")
        print("Output:", row["output"])
else:
    print("\n[PASS] Validation score is within acceptable bounds. Ready to run test inference.")

### 6. RUN TEST MODE
Generates responses over the official `test.csv` using MBR decoding.

In [ ]:
test_path = data_utils.find_data_file("test.csv")
test_df = pd.read_csv(test_path)

print("Starting final test inference...")
test_preds_df = inference_utils.run_inference(model, tokenizer, test_df, k=4)

# Ensure submission dataframe matches exactly
submission_df = test_preds_df.copy()
submission_df.to_csv("/kaggle/working/submission.csv", index=False)
print("Saved final submission to /kaggle/working/submission.csv")

### 7. Submission Integrity Verification

In [ ]:
submission_path = "/kaggle/working/submission.csv"
sub_df = pd.read_csv(submission_path)

print("=== Submission Verification ===")
print(f"Row count matches test: {len(sub_df) == len(test_df)} (Count: {len(sub_df)})")
print(f"Column names match: {list(sub_df.columns) == ['id', 'output']}")

null_count = sub_df["output"].isna().sum() + sub_df["output"].str.strip().eq("").sum()
print(f"Null/Empty predictions: {null_count}")

if null_count > 0:
    print("Fallback mechanism: filling empty cells using default doctor response...")
    fallback = "অনুগ্রহ করে আপনার লক্ষণগুলোর অবনতি ঘটলে সরাসরি একজন বিশেষ ডাক্তারের সাথে যোগাযোগ করুন এবং সঠিক পরীক্ষা করান।"
    sub_df["output"] = sub_df["output"].fillna(fallback).replace("", fallback)
    sub_df.to_csv(submission_path, index=False)
    print("Corrected submission saved.")
    
print("\nFirst 5 rows of submission:")
print(sub_df.head(5).to_string(max_colwidth=100))